* EX2.5 (2 points): Using either the parser function `extract_spectroscopic_data_from_gauss_file` or `extract_spectroscopic_data_from_gauss_file_regex` from Section 2.3 in lecture L06:
  - Extract all spectroscopic data from the Gaussian `.log` files in the folder `simulation_text_files` from lecture L06 and store it in a single pandas DataFrame.
  - Loop over all files in `simulation_text_files`, and only process files with `.log` extension.
  - Modify the parser to include a column `filename` in the DataFrame, containing the name of the file (without path and without extension) from which each row was extracted.
  - Filter the DataFrame to keep **only the three rows with the lowest values of `Frequencies` for each file**.
  - HINT: you may need to group the data by `filename`
  - Create a folder `processed_spectroscopic_data` and store the filtered data in a CSV file named `spectroscopic_data.csv`.
  
Solution: 
|   | Symmetry | Frequencies | Red. masses | Frc consts | IR Inten | filename                    | Raman Activ | Depolar (P) | Depolar (U) |
|---|----------|-------------|-------------|------------|----------|-----------------------------|-------------|-------------|-------------|
| 1 | A        | 30.9016     | 8.186       | 0.0046     | 0.0766   | first_gaussian_example      | 38.3341     | 0.75        | 0.8571      |
| 2 | A        | 45.1395     | 3.7157      | 0.0045     | 1.1354   | first_gaussian_example      | 7.563       | 0.5874      | 0.7401      |
| 3 | A        | 50.8418     | 7.2219      | 0.011      | 0.6991   | first_gaussian_example      | 53.7174     | 0.75        | 0.8571      |
| 4 | A        | 242.6076    | 30.4472     | 1.0559     | 0.0      | second_gaussian_example     |             |             |             |
| 5 | A        | 253.8678    | 41.4663     | 1.5746     | 0.0576   | second_gaussian_example     |             |             |             |
| 6 | A        | 291.0721    | 28.2614     | 1.4107     | 2.7388   | second_gaussian_example     |             |             |             |



In [24]:
# Import libraries
import os
from pathlib import Path
import pandas as pd

In [41]:
"""
Extract spectroscopic data from Gaussian log files, keep the three lowest
frequency modes for each file, reorder the columns, and save the processed
data as a CSV file.

Inputs:
    input_dir: Directory containing the Gaussian .log files.
    output_dir: Directory where the processed CSV file will be saved.

Output:
    A CSV file named "spectroscopic_data.csv" containing the three lowest
    frequency entries for each processed Gaussian log file.
"""

# Define the input and output directories
# Create the output directory if it does not already exist
input_dir = Path('simulation_text_files')
output_dir = Path('processed_spectroscopic_data')
output_dir.mkdir(parents=True, exist_ok=True)

# Create a list to store the data extracted from each file
all_data = []

# Iterate through the input directory and process only .log files
for file_path in input_dir.iterdir():
    if file_path.is_file() and file_path.suffix == '.log':

        # Extract the spectroscopic data from the current Gaussian log file
        # (the parser already adds the filename column internally)
        df_file = extract_spectroscopic_data_from_gauss_file(file_path)

        # Store the extracted data
        all_data.append(df_file)

# Combine the data from all files into a single DataFrame
combined_df = pd.concat(all_data)

# Keep only the three rows with the lowest frequency values for each file
filtered_df = (
    combined_df.sort_values(by='Frequencies')
    .groupby('filename', as_index=False)
    .head(3)
)

# Reorder the columns to match the expected output format
expected_columns = [
    'Symmetry', 'Frequencies', 'Red. masses', 'Frc consts',
    'IR Inten', 'filename', 'Raman Activ', 'Depolar (P)', 'Depolar (U)'
]
filtered_df = filtered_df[expected_columns]

# Reset the index so that it starts at 1 and is consecutive
filtered_df.index = range(1, 7)

# Define the output CSV file path
output_file = output_dir / 'spectroscopic_data.csv'

# Save the filtered data to a CSV file
filtered_df.to_csv(output_file, index=False)

In [28]:
def extract_spectroscopic_data_from_gauss_file(filename):
    """
    Extract spectroscopic data from a Gaussian `.log` file and return it
    as a pandas DataFrame.

    Args:
        filename (str or pathlib.Path): Path to the `.log` file.

    Returns:
        pandas.DataFrame: DataFrame with the spectroscopic data.
    """
    # Ensure Path object
    filename = Path(filename)

    df = pd.DataFrame([])
    block_started = False

    with filename.open(mode="r") as file:
        for line in file:
            # Detect start of relevant block
            if "and normal coordinates:" in line:
                block_started = True # this controls when to start storing data
                data_dict = {}
                continue # this makes the for loop progress to the next item

            if block_started:
                # Detect end of block
                if "- Thermochemistry -" in line:
                    break # this finishes the for loop, even if there are more items in the iterator

                splitted_line = line.split()

                match len(splitted_line):
                    case 3:
                        if splitted_line[0].isnumeric():
                            label = "index"
                        else:
                            label = "Symmetry"
                    case 5:
                        label = splitted_line[0]
                    case 6:
                        label = splitted_line[0] + " " + splitted_line[1]
                    case 0:
                        continue
                    case _:
                        if splitted_line[0] == "Atom":
                            index = data_dict.pop("index")
                            df = add_data_to_df(
                                df=df,
                                data_dict=data_dict,
                                index=index
                            )
                            data_dict = {}
                        continue

                # Convert numeric data
                data = splitted_line[-3:]
                if label not in ["index", "Symmetry"]:
                    data = [float(item) for item in data]

                data_dict[label] = data
    df['filename'] = filename.stem
    return df

In [27]:
def add_data_to_df(df, data_dict, index):
    """
    Adds the data contained in data_dict to a pandas DataFrame.

    Args:
        df (pd.DataFrame): Original DataFrame.
        data_dict (dict): Data to be added.
        index (list): Index where the data should be added.

    Returns:
        pd.DataFrame: Updated DataFrame.
    """
    new_data = pd.DataFrame(data_dict, index=index)
    if df.empty:
        df = new_data
    else:
        df = pd.concat([df, new_data])
    return df